# Sentiment Analysis using Recurrent Neural Networks (RNN) on IMDB Reviews

### What was learned & implemented in this notebook:
1. **NLP & Sentiment Classification**:
   - Analyzed sentiment classification (Positive vs Negative reviews) over a dataset of 50,000 text records.
2. **Data Vectorization & Preprocessing**:
   - Extracted features and converted text into structured representations using TF-IDF sparse matrix vectorization, subsequently converting it to dense float tensors.
3. **Recurrent Neural Network Architecture**:
   - Configured custom PyTorch `nn.Module` subclass wrapping PyTorch's recurrent unit `nn.RNN`.
   - Initialized context state dimension `h0` dynamically and unsqueezed features to inject required sequence dimensions `(batch_size, sequence_length, features)`.
4. **Optimization & Binary Classification Metrics**:
   - Squashed linear output scores using a Sigmoid activation and evaluated using Binary Cross Entropy Loss (`nn.BCELoss`).
   - Trained the network over 10 epochs using the Adam optimizer, achieving a solid test accuracy of **~81.0%**.

In [11]:
import pandas as pd

In [12]:
df = pd.read_csv("IMDB Dataset.csv")

In [13]:
df.shape

(50000, 2)

In [14]:
df.head(15)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
5,"Probably my all-time favorite movie, a story o...",positive
6,I sure would like to see a resurrection of a u...,positive
7,"This show was an amazing, fresh & innovative i...",negative
8,Encouraged by the positive comments about this...,negative
9,If you like original gut wrenching laughter yo...,positive


In [15]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [16]:
df.drop_duplicates(inplace = True)

In [17]:
df.shape

(49582, 2)

### TEXT PREPROCESSING

### 1. Convert to LowerCase

In [18]:
df["review"]= df["review"].str.lower()
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


### 2. Remove Url

In [19]:
import re

def remove_url(text):
    text = re.sub(r"http\S+","",text)
    return text
df["review"]= df["review"].apply(remove_url)

In [20]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


### 3. Remove Punc

In [21]:
def remove_punctuation(text):
    text = re.sub(r"[^A-Za-z0-9\s]","",text)
    return text
df["review"]= df["review"].apply(remove_punctuation)

In [22]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 4. removing HTML

In [23]:
def remove_html(text):
    text = re.sub("<.?>","",text)
    return text
df["review"]= df["review"].apply(remove_html)

In [24]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 5. Removing Stopwords

In [25]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /Users/pashin/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/pashin/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/pashin/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [26]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [27]:
def reomve_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word  in stop_words:
            text = text.replace(word,"")

    return text 
df["review"]= df["review"].apply(reomve_stopwords)


In [28]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6. Stemming

In [29]:
from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer()
    stemmed_words =[]
    tokens =word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
        
df["review"]= df["review"].apply(reomve_stopwords)  

In [30]:
df.head()

,review,sentiment
0,e revewers nte wtchg 1 oz epoe hooke ...,positive
1,wderful ltle prducti br br filming technique...,positive
2,hugh h werful w pen me h ummer weeken n...,positive
3,bcy fly e boy jke hk zobe cloe pn f...,negative
4,peer me love i vu unng film wch mr mei ...,positive


### 7. Encoding 

In [31]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["sentiment"]= le.fit_transform(df["sentiment"])


In [32]:
y = df["sentiment"]

In [33]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8. Vectorization

In [34]:
df.head()

,review,sentiment
0,e revewers nte wtchg 1 oz epoe hooke ...,1
1,wderful ltle prducti br br filming technique...,1
2,hugh h werful w pen me h ummer weeken n...,1
3,bcy fly e boy jke hk zobe cloe pn f...,0
4,peer me love i vu unng film wch mr mei ...,1


In [35]:
from  sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features= 2000)
X = tf.fit_transform(df["review"])

### Dataset & DataLoader

In [36]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train, y_test = train_test_split(X,y,random_state =42, test_size = 0.2)

In [37]:
X_train.shape

(39665, 2000)

In [38]:
X_test.shape

(9917, 2000)

In [39]:
import torch
from torch.utils.data import DataLoader, TensorDataset

In [40]:
X_train = X_train.toarray()
X_test = X_test.toarray()


In [41]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [42]:
train_loader = DataLoader(train_set,shuffle = True,batch_size=64)
test_loader = DataLoader(test_set,shuffle = True,batch_size=64)

In [43]:
import torch.nn as nn
import torch.optim as optim

In [53]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size =120,num_layers= 1):
        super().__init__()
        self.hidden_size= hidden_size
        self.num_layers = num_layers

        #RNN LAYER
        self.rnn = nn.RNN(input_size,hidden_size,num_layers,batch_first =True)
        #FC LAYER
        self.fc = nn.Linear(hidden_size, 1)


    def forward(self,x):
        # optional => shape (num of layers, batch size, hidden size)
        h0= torch.zeros(self.num_layers,x.size(0),self.hidden_size)

        out,_ = self.rnn(x,h0)
         # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
            # 2nd value = final hidden state of last timestep
        out = self.fc(out[:,-1,:])
        return out
            
        

In [54]:
input_size = X_train.shape[1]
model = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

### TRAIN RNN

In [55]:
epochs = 10 
for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()
        Xb = Xb.unsqueeze(1) # Inserts a sequence dimension of length 1 into tensor shape: from (64, 5000) to (64, 1, 5000) ((batch_size,sequence_length, features)).
        outputs = model(Xb) # returns batch_size,1
        outputs = torch.sigmoid(outputs.squeeze())# (batch_size,) => probability
        loss = criterion(outputs,yb) #Computes binary cross-entropy loss between predicted probabilities and ground truth yb.
        loss.backward()
        optimizer.step()

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")
        
        
        

epoch = 1/10 and loss = 0.5482604503631592
epoch = 2/10 and loss = 0.2752959132194519
epoch = 3/10 and loss = 0.48142772912979126
epoch = 4/10 and loss = 0.4717702269554138
epoch = 5/10 and loss = 0.3912361264228821
epoch = 6/10 and loss = 0.32866352796554565
epoch = 7/10 and loss = 0.38706547021865845
epoch = 8/10 and loss = 0.27759167551994324
epoch = 9/10 and loss = 0.3915424644947052
epoch = 10/10 and loss = 0.3614672124385834


In [57]:
model.eval()
with torch.no_grad():
    correct_vals = 0
    tot_vals = 0 

    for Xb,yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze())>0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"Accuracy = {correct_vals/tot_vals*100}")

Accuracy = 80.98215186044166
